raw text files
  → 1. stream      read piece by piece, never the whole file at once
  → 2. normalize   make equivalent text identical (Unicode, whitespace)
  → 3. exact dedup  drop documents that are byte-for-byte copies
  → 4. near dedup   drop documents that are almost copies (MinHash)
  → 5. shard        write clean output into fixed-size files
clean shards → ready for training

In [61]:
import ssl
import urllib.request
import unicodedata
import certifi
import os
import re
import hashlib

urls = [
    "https://www.gutenberg.org/files/11/11-0.txt",
    # "https://www.gutenberg.org/files/1342/1342-0.txt",
    # "https://www.gutenberg.org/files/84/84-0.txt",
    # "https://www.gutenberg.org/files/1661/1661-0.txt",
]


def load_training_text(urls=urls):
    context = ssl.create_default_context(cafile=certifi.where())
    texts = []
    for url in urls:
        response = urllib.request.urlopen(url, context=context, timeout=30)
        text = response.read().decode("utf-8-sig")
        texts.append(text)
    training_text = "\n\n".join(texts) + "<|endoftext|>"
    return training_text

In [62]:
data = load_training_text()
with open("./large_file.txt", "w", encoding="utf-8") as file:
    file.write(data)

In [63]:
def read_files(file_paths):
    for path in file_paths:
        with open(path, "r") as file:
            yield file.read()

In [64]:
def read_line(path:str):
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            yield line

In [ ]:
def normalize(text):
    text = unicodedata.normalize("NFC", text)
    text = text.translate(str.maketrans({
        "‘": "'",
        "’": "'",
        "“": '"',
        "”": '"',
    }))
    text = text.translate(str.maketrans({
        "–": "-",
        "—": "-",
        "−": "-",
    }))
    text = text.replace("\u00A0", " ")
    text = text.replace("\u200B", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [73]:
def generate_hash(text):
    return hashlib.sha256(text.encode("utf-8"))

In [ ]:
def remove_boilerplate(text):
    # Remove unwanted headers, footers, and page numbers
    ...

In [76]:
def quality_filter(text,min_words=30,min_alpha_ratio=0.5,max_repeated_ratio=0.5,max_long_word_ratio=0.2):
    if not text.strip():
        return False
    words = text.split()
    if len(words) < min_words:
        return False
    non_space_chars = [c for c in text if not c.isspace()]
    alpha_chars = sum(c.isalpha() for c in non_space_chars)
    alpha_ratio = alpha_chars / len(non_space_chars)
    if alpha_ratio < min_alpha_ratio:
        return False
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        repeated_ratio = 1 - (len(set(lines)) / len(lines))
        if repeated_ratio > max_repeated_ratio:
            return False
    long_words = sum(len(word) > 30 for word in words)
    long_word_ratio = long_words / len(words)
    if long_word_ratio > max_long_word_ratio:
        return False
    return True

In [ ]:
seen = set()
dir = "D:/docs"
paths = [os.path.join(dir,path) for path in os.listdir(path=dir)if os.path.isfile(os.path.join(dir, path))]
for text in read_files(paths):
    norm_text = normalize(text)
    norm_text = remove_boilerplate(norm_text)
    if not quality_filter(text):
        continue
    hash_value = generate_hash(norm_text).hexdigest()
    if hash_value in seen:
        continue
    seen.add(hash_value)
    print(hash_value)
    

7d1d3a71a80fb0b98e9d088216ac535e392d3126954fc9c112d0518acb77acaa
ba9426cad41f858e561297195d0234b968e57b1f36a0070f78c5177292bdafcc
05cb0cc972b9c181646115a0a9463af06ee7ba27742ef1e464ec93af93a190d0
e22d5c27781afe88532bb21b69301f7b98521b09dfe8706033961a3c2064c880


In [68]:
dir = "D:/docs"
for path in os.listdir(path=dir):
    file_path = os.path.join(dir,path)
    for line in read_line(file_path):
        norm_line = normalize(line)
        print(norm_line)
        break
    break
    # for line in read_line(file_path):
    #     norm_line = normalize(line)
    #     print(norm_line)
    #     break
    # break

RIVER RESTORATION FIELD REPORT


In [69]:
def stream_doc(path:str):
    with open(path, "r", encoding="utf-8") as file:
        yield file.read()
        # for line in file:
        #     yield line
    



In [70]:
with open("large_file.txt", "r", encoding="utf-8") as file:
    while chunk := file.read(1024):
        chunk.encode("utf-8")
        
